# Rank Points Diff Comparison

Confronto minimale tra differenza lineare dei rank points e differenza logaritmica dei rank points, con baseline semplice e LightGBM a una sola feature.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

import lightgbm as lgb


In [2]:
PROCESSED_DIR = Path("../data/processed")

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")


In [3]:
TARGET_COL = "target"


In [6]:
def evaluate_single_feature(feature_col, baseline_fn):
    X_train = train_df[[feature_col]].copy()
    y_train = train_df[TARGET_COL]

    X_test = test_df[[feature_col]].copy()
    y_test = test_df[TARGET_COL]

    fill_value = X_train[feature_col].median()
    X_train[feature_col] = X_train[feature_col].fillna(fill_value)
    X_test[feature_col] = X_test[feature_col].fillna(fill_value)

    model = lgb.LGBMClassifier(
        objective="binary",
        metric="binary_logloss",
        n_estimators=500,
        learning_rate=0.03,
        max_depth=3,
        num_leaves=7,
        min_child_samples=30,
        subsample=1.0,
        colsample_bytree=1.0,
        random_state=42
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[
            lgb.early_stopping(30),
            lgb.log_evaluation(50)
        ]
    )

    test_pred_proba = model.predict_proba(X_test)[:, 1]
    test_pred = (test_pred_proba >= 0.5).astype(int)

    baseline_score = X_test[feature_col]
    baseline_proba = baseline_fn(baseline_score)
    baseline_pred = (baseline_proba >= 0.5).astype(int)

    return {
        "feature": feature_col,
        "baseline_accuracy": accuracy_score(y_test, baseline_pred),
        "baseline_log_loss": log_loss(y_test, baseline_proba),
        "baseline_auc": roc_auc_score(y_test, baseline_proba),
        "lgbm_accuracy": accuracy_score(y_test, test_pred),
        "lgbm_log_loss": log_loss(y_test, test_pred_proba),
        "lgbm_auc": roc_auc_score(y_test, test_pred_proba),
    }


In [7]:
log_results = evaluate_single_feature(
    "rank_log_points_diff",
    baseline_fn=lambda score: 1 / (1 + np.exp(-score)),
)


[LightGBM] [Info] Number of positive: 817047, number of negative: 817047
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001629 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 1634094, number of used features: 1
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 30 rounds
[50]	valid_0's binary_logloss: 0.641586
[100]	valid_0's binary_logloss: 0.640364
Early stopping, best iteration is:
[83]	valid_0's binary_logloss: 0.640275
[LightGBM] [Info] Number of positive: 817047, number of negative: 817047
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001705 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `f

In [8]:
pd.DataFrame([
    log_results,
]).set_index("feature")


,baseline_accuracy,baseline_log_loss,baseline_auc,lgbm_accuracy,lgbm_log_loss,lgbm_auc
feature,,,,,,
rank_points_diff,0.652891,0.680664,0.677082,0.652891,0.640275,0.678639
rank_log_points_diff,0.652891,0.669041,0.712988,0.652891,0.619548,0.713068
